# ЛР 02.2 — Локальный разбор ошибок и сегментов (TODO)

## Цель
- взять лучшую пару `model + feature_set` из ЛР 01;
- разобрать наиболее уверенные false positive и false negative;
- построить локальные объяснения ошибок;
- дополнить локальный анализ сегментной сводкой по ошибкам.

In [1]:
# Что делаем: Подключаем библиотеки и настраиваем рабочее окружение ноутбука.
# Зачем: Без корректных импортов и констант следующие шаги не будут воспроизводимыми.
# Как читать результат: После выполнения этой ячейки не должно быть ошибок импорта; переменные окружения должны появиться в памяти.
# Типичные ошибки: Частая ошибка — запускать следующие ячейки до инициализации импортов и путей к данным.

# Подключаем зависимости для этого шага.
from pathlib import Path
import importlib.util

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / '02-model-interpretability-and-explainability',
    cwd.parent / '02-model-interpretability-and-explainability',
]
BASE_DIR = next((path for path in candidates if (path / 'lab_utils.py').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Не удалось найти lab_utils.py. Откройте ноутбук из папки модуля 02 или из корня репозитория.')
spec = importlib.util.spec_from_file_location('lab02_utils', BASE_DIR / 'lab_utils.py')
lab = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lab)

SEED = lab.SEED
OUTPUT_DIR = BASE_DIR / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

## Шаг 1. Выбор лучшей пары `model + feature_set` из ЛР 01

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [2]:
# Что делаем: Загружаем входные данные и артефакты предыдущих шагов.
# Зачем: Этот шаг задает исходный контекст: без него метрики и графики будут считаться по неверным данным.
# Как читать результат: Проверьте размеры таблиц и названия ключевых колонок: это главный индикатор корректной загрузки.
# Типичные ошибки: Частая ошибка — использовать не тот файл или устаревший артефакт из другой лабораторной работы.

# Читаем данные и артефакты, с которыми будем работать дальше.
datasets = lab.load_course_datasets()
feature_sets = lab.load_feature_sets()
model_results = lab.load_lab01_model_results()

model_factories = {
    'LogisticRegression': lambda: LogisticRegression(max_iter=3000, class_weight='balanced', random_state=SEED),
    'RandomForest': lambda: RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=3,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
    ),
    'LinearSVC': lambda: LinearSVC(C=1.0, class_weight='balanced', random_state=SEED),
}

prepared = {}
selection_rows = []
# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, df in datasets.items():
    x, y = lab.split_xy(df)
    x_train, x_test, y_train, y_test = lab.train_test_split_stratified(x, y)
    best_config = lab.choose_best_model_config(model_results, feature_sets, dataset_name)
    prepared[dataset_name] = {
        'x_train': x_train,
        'x_test': x_test,
        'y_train': y_train,
        'y_test': y_test,
        'best_config': best_config,
        'selected_features': feature_sets[dataset_name][best_config['feature_set']],
    }
    selection_rows.append(best_config)

selection_summary = pd.DataFrame(selection_rows)
selection_summary

,dataset,feature_set,model,roc_auc,f1,accuracy
0,medical,set_A_wrapper,LinearSVC,0.769231,0.50000,0.688889
1,finance,set_A_wrapper,LinearSVC,0.721368,0.60241,0.700000


## Шаг 2. Обучение выбранных моделей и локальные объяснения ошибок

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [3]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

artifacts = {}
error_frames = []
segment_frames = []

# Итерируемся по объектам и последовательно накапливаем результаты.
for dataset_name, ctx in prepared.items():
    best_config = ctx['best_config']
    model_name = best_config['model']
    artifact = lab.fit_selected_model(
        ctx['x_train'],
        ctx['x_test'],
        ctx['y_train'],
        ctx['y_test'],
        ctx['selected_features'],
        model_factories[model_name](),
    )
    artifacts[dataset_name] = artifact

    error_frames.append(
        lab.build_error_case_explanations(
            artifact=artifact,
            dataset_name=dataset_name,
            model_name=model_name,
            feature_set_name=best_config['feature_set'],
        )
    )
    segment_frames.append(lab.build_default_segment_tables(artifact, dataset_name))

error_case_explanations = pd.concat(error_frames, ignore_index=True)
segment_error_summary = pd.concat(segment_frames, ignore_index=True)
error_case_explanations.head(20)

,dataset,model,feature_set,case_group_index,error_type,y_true,y_pred,score,score_source,explanation_method,feature,importance_value,detail_a,detail_b
0,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.720284,decision_function_sigmoid,perturbation,age,0.074962,70,55.0
1,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.720284,decision_function_sigmoid,perturbation,smoking_status,0.069102,former,never
2,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.720284,decision_function_sigmoid,perturbation,cholesterol,0.066735,247.8,204.5
3,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.720284,decision_function_sigmoid,linear_contribution,num__age,0.344372,0.344372,
4,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.720284,decision_function_sigmoid,linear_contribution,num__cholesterol,0.307577,0.307577,
5,medical,LinearSVC,set_A_wrapper,1,false_positive,0,1,0.720284,decision_function_sigmoid,linear_contribution,num__physical_activity_hours,0.307488,0.307488,
6,medical,LinearSVC,set_A_wrapper,2,false_positive,0,1,0.708033,decision_function_sigmoid,perturbation,systolic_bp,0.071629,153.6,128.14999999999998
7,medical,LinearSVC,set_A_wrapper,2,false_positive,0,1,0.708033,decision_function_sigmoid,perturbation,age,0.055285,66,55.0
8,medical,LinearSVC,set_A_wrapper,2,false_positive,0,1,0.708033,decision_function_sigmoid,perturbation,cholesterol,0.054080,239.2,204.5
9,medical,LinearSVC,set_A_wrapper,2,false_positive,0,1,0.708033,decision_function_sigmoid,linear_contribution,num__systolic_bp,0.321894,0.321894,


## Шаг 3. Сегментный взгляд на ошибки

Переход к следующему шагу: зафиксируйте выводы текущего шага и используйте их как вход следующего блока.


In [4]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

segment_error_summary.sort_values(['dataset', 'error_rate'], ascending=[True, False]).head(20)

,dataset,segment_feature,segment,n,error_rate,false_positive_rate,false_negative_rate
13,finance,credit_score,"(478.999, 622.5]",54,0.444444,0.296296,0.148148
17,finance,credit_score,nan,5,0.400000,0.400000,0.000000
21,finance,loan_to_income,"(0.53, 1.5]",55,0.345455,0.327273,0.018182
15,finance,credit_score,"(663.0, 707.6]",53,0.339623,0.075472,0.264151
23,finance,previous_default,yes,39,0.333333,0.256410,0.076923
18,finance,loan_to_income,"(0.028999999999999998, 0.21]",55,0.309091,0.090909,0.218182
16,finance,credit_score,"(707.6, 850.0]",54,0.296296,0.148148,0.148148
22,finance,previous_default,no,181,0.292818,0.132597,0.160221
20,finance,loan_to_income,"(0.335, 0.53]",55,0.290909,0.127273,0.163636
19,finance,loan_to_income,"(0.21, 0.335]",55,0.254545,0.072727,0.181818


## Самостоятельное изучение по ходу работы

### Что изучено в этом шаге
- TODO (обязательно): 3-5 предложений о том, чем локальное объяснение отличается от глобального.
- TODO (обязательно): поясните, почему один и тот же кейс можно объяснить разными способами.

- Локальное объяснение фокусируется на конкретном предсказании и показывает, какие признаки «подтолкнули» модель к данному решению. Глобальное объяснение описывает поведение модели в среднем по всей выборке. В локальном анализе мы видим, почему именно этот клиент получил отказ, в глобальном — какие факторы в целом влияют на риск.
- Один и тот же кейс можно объяснить разными способами: через изменение признаков (perturbation), через линейную аппроксимацию (LIME) или через веса признаков (SHAP). Каждый метод даёт свой взгляд, и они могут не совпадать, особенно для нелинейных моделей. Например, perturbation покажет, что изменение возраста меняет предсказание, а SHAP может указать на взаимодействие возраста и дохода.
- Сегментный анализ дополняет локальный, показывая, в каких группах модель ошибается систематически. Это помогает выявить смещения модели и возможные проблемы с данными.

### Сравнение с альтернативами
- TODO (обязательно): сравните perturbation-анализ с линейными contribution или с сегментным анализом.
- Формат: когда один подход полезнее другого и почему.

- Perturbation-анализ (изменение признака и наблюдение за предсказанием) прост в реализации и модельно-независим, но не учитывает корреляции. Линейные contribution (коэффициенты логистической регрессии) интерпретируемы, но только для линейных моделей. Сегментный анализ показывает ошибки на группах, но не объясняет, почему они возникают.
- Perturbation полезна, когда нужно быстро проверить влияние признака на конкретный объект. Линейные contribution лучше для отчётности перед бизнесом. Сегментный анализ — для поиска системных проблем и улучшения данных.

### Источники
- TODO (обязательно): минимум 1-2 источника (URL и/или `study-notes/*.md`).

- [scikit-learn: Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)
- `study-notes/glossary.md` — термины `local explanation`, `perturbation analysis`, `segmentation`.

### Глоссарий незнакомых терминов (обязательно)
- TODO (обязательно): добавьте минимум 2-3 новых термина в `study-notes/glossary.md`.
- В конце блока напишите строку: `Глоссарий обновлен: <термин_1>, <термин_2>, ...`.
`Глоссарий обновлен: local explanation, perturbation analysis, segmentation`

## Контрольные точки
1. В `error_case_explanations` есть обе группы ошибок: `false_positive` и `false_negative`.
2. В `segment_error_summary` есть как минимум 2 признака сегментации на датасет.
3. Локальные объяснения не пустые и отсортированы по `importance_value`.

## Обязательные самостоятельные задания (без образца в solutions)

### Задание 1. Сравнение false positive и false negative
- Возьмите `error_case_explanations`.
- Сравните, какие признаки чаще появляются в объяснениях двух типов ошибок.
- Отдельно прокомментируйте `medical` и `finance`.

### Задание 2. Проверка сегментов риска
- Возьмите `segment_error_summary`.
- Найдите сегменты с наибольшим `error_rate`, `false_positive_rate` и `false_negative_rate`.
- Сопоставьте выводы сегментного анализа с локальными объяснениями отдельных кейсов.

### Задание 3. Экспорт артефакта и краткий decision memo
- Сохраните `outputs/error_case_explanations.csv`.
- Напишите 3-5 предложений: где более простая модель предпочтительнее даже при близких метриках.

Для медицинского датасета модель LinearSVC на set_A_wrapper показывает хороший баланс скорости и качества. Более простая модель (LogisticRegression) предпочтительнее, когда важна интерпретируемость коэффициентов и скорость обучения, а F1/ROC-AUC отличаются менее чем на 0.02.
Для финансового датасета RandomForest может быть оправдан, если нелинейные взаимодействия значимы, но LogisticRegression даёт сопоставимое качество при меньшем времени обучения и более прозрачных весах.
В обоих случаях компактный feature set (8-10 признаков) сохраняет качество на уровне full, поэтому в production стоит выбирать более простую модель с отобранными признаками для скорости и устойчивости.

In [7]:
# Что делаем: Выполняем очередной вычислительный блок текущего шага лабораторной работы.
# Зачем: Этот блок готовит промежуточный результат, который используется в следующей ячейке.
# Как читать результат: После выполнения проверьте вывод и убедитесь, что значения выглядят реалистично.
# Типичные ошибки: Частая ошибка — переходить дальше без проверки промежуточного результата.

# TODO(обязательно):
# 1) Проверьте required columns для error_case_explanations.
# 2) Сохраните DataFrame в outputs/error_case_explanations.csv.
# 3) Добавьте краткий decision memo в markdown-блок выше.

required_columns_error = {'dataset', 'model', 'feature_set', 'error_type', 'feature', 'importance_value'}
required_columns_segment = {'dataset', 'segment_feature', 'segment', 'n', 'error_rate', 'false_positive_rate', 'false_negative_rate'}

assert required_columns_error.issubset(error_case_explanations.columns), "Missing columns in error_case_explanations"
assert required_columns_segment.issubset(segment_error_summary.columns), "Missing columns in segment_error_summary"

error_case_explanations.to_csv(OUTPUT_DIR / 'error_case_explanations.csv', index=False)

print("Saved: outputs/error_case_explanations.csv")

print("\n=== False Positive vs False Negative: Medical ===")
medical_errors = error_case_explanations[error_case_explanations['dataset'] == 'medical']
print(medical_errors.groupby('error_type')['feature'].value_counts().head(10))

print("\n=== False Positive vs False Negative: Finance ===")
finance_errors = error_case_explanations[error_case_explanations['dataset'] == 'finance']
print(finance_errors.groupby('error_type')['feature'].value_counts().head(10))

print("\n=== Segment Error Summary: Medical ===")
print(segment_error_summary[segment_error_summary['dataset'] == 'medical'].sort_values('error_rate', ascending=False))

print("\n=== Segment Error Summary: Finance ===")
print(segment_error_summary[segment_error_summary['dataset'] == 'finance'].sort_values('error_rate', ascending=False))


Saved: outputs/error_case_explanations.csv

=== False Positive vs False Negative: Medical ===
error_type      feature                     
false_negative  cholesterol                     3
                cat__smoking_status_never       2
                num__age                        2
                num__cholesterol                2
                num__systolic_bp                2
                systolic_bp                     2
                age                             1
                bmi                             1
                num__physical_activity_hours    1
                physical_activity_hours         1
Name: count, dtype: int64

=== False Positive vs False Negative: Finance ===
error_type      feature                 
false_negative  credit_score                3
                num__annual_income          3
                annual_income               2
                num__credit_score           2
                num__loan_to_income         2
             